# 07 Political-Corruption Attention Relative To Total News Coverage

This notebook loads the classified political-corruption articles and merges them with **total news coverage** denominators by country-week/month.

The key measure is:

```text
relative_attention = predicted political-corruption articles / total news articles
```

This is different from the cleaned corruption-query corpus denominator. Do not use `denominator_country_week.csv` or `denominator_country_month.csv` for the main relative-attention measure here; those files only describe the corruption-query corpus.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

PIPELINE_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline"
)
CLASSIFIER_DIR = PIPELINE_DIR / "silver_classifier"
CLASSIFIED_DIR = CLASSIFIER_DIR / "classified_country_files"
FIGURE_DIR = PIPELINE_DIR / "attention_figures"
TABLE_DIR = PIPELINE_DIR / "attention_tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Local fallback paths. If these do not exist, the notebook tries WebDAV below.
TOTAL_WEEK_PATH = PIPELINE_DIR / "total_news_coverage_week.csv"
TOTAL_MONTH_PATH = PIPELINE_DIR / "total_news_coverage_month.csv"

# Mounted WebDAV source for total-news denominator files.
# This folder contains one weekly count CSV per country, e.g. Bulgaria_weekly_count.csv.
TOTAL_COVERAGE_MOUNT_DIR = Path(
    "/home/akroon/webdav/ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/"
    "weekly_counts_total_coverage"
)

# Research Drive/WebDAV API fallback for the same folder.
TOTAL_COVERAGE_RD_DIR = (
    "ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/"
    "weekly_counts_total_coverage"
)

COUNTRY_ORDER = [
    "Bulgaria",
    "France",
    "Hungary",
    "Italy",
    "Netherlands",
    "Serbia",
    "Sweden",
    "Ukraine",
    "United_Kingdom",
]

COUNTRY_LABELS = {
    "United_Kingdom": "United Kingdom",
}

# Monthly is usually clearer for manuscript figures; weekly is also computed if a weekly denominator exists.
PLOT_LEVEL = "month"  # "month" or "week"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#333333",
    "grid.color": "#dddddd",
    "grid.linewidth": 0.7,
})

print(f"Pipeline dir: {PIPELINE_DIR}")
print(f"Classified dir: {CLASSIFIED_DIR}")
print(f"Local total-news weekly denominator path:  {TOTAL_WEEK_PATH}")
print(f"Local total-news monthly denominator path: {TOTAL_MONTH_PATH}")
print(f"Mounted total-news denominator folder:    {TOTAL_COVERAGE_MOUNT_DIR}")
print(f"WebDAV total-news denominator folder:      {TOTAL_COVERAGE_RD_DIR}")

## 2. Load Total-News Denominators

Provide total-news denominator files before running the analysis. These should count all news coverage for the selected outlets/countries/time period, not only articles retrieved with corruption keywords.

Default expected paths:

```text
political_corruption_pipeline/total_news_coverage_week.csv
political_corruption_pipeline/total_news_coverage_month.csv
```

If only the weekly file exists, the notebook derives monthly totals from week-start dates.

In [ ]:
COUNT_COLUMN_CANDIDATES = [
    "total_news_articles",
    "total_articles",
    "total_coverage",
    "n_articles",
    "count",
    "n",
    "weekly_count",
]

PERIOD_COLUMN_CANDIDATES = {
    "week": ["week", "week_start", "date_week", "period", "date", "start_date"],
    "month": ["month", "month_start", "date_month", "period", "date"],
}

FILENAME_COUNTRY_MAP = {
    "UK": "United_Kingdom",
    "United Kingdom": "United_Kingdom",
    "United_Kingdom": "United_Kingdom",
}


def country_from_filename(filename):
    stem = Path(filename).stem
    stem = stem.replace("_weekly_count", "").replace("_week_count", "")
    stem = stem.replace("_weekly_counts", "").replace("_week_counts", "")
    return FILENAME_COUNTRY_MAP.get(stem, stem)


def standardize_total_coverage(data, period_column, source_name="", country=None):
    data = data.copy()

    if period_column not in data.columns:
        period_match = next(
            (column for column in PERIOD_COLUMN_CANDIDATES[period_column] if column in data.columns),
            None,
        )
        if period_match is None:
            raise ValueError(
                f"Could not find a {period_column!r} column in {source_name}. "
                f"Expected one of: {PERIOD_COLUMN_CANDIDATES[period_column]}"
            )
        data = data.rename(columns={period_match: period_column})

    count_column = next((column for column in COUNT_COLUMN_CANDIDATES if column in data.columns), None)
    if count_column is None:
        numeric_candidates = [
            column for column in data.columns
            if column != period_column and pd.api.types.is_numeric_dtype(data[column])
        ]
        if len(numeric_candidates) == 1:
            count_column = numeric_candidates[0]
            print(f"Using numeric count column {count_column!r} for {source_name}")
        else:
            raise ValueError(
                f"Could not find a count column in {source_name}. Expected one of: {COUNT_COLUMN_CANDIDATES}"
            )

    if "country" not in data.columns:
        if country is None:
            raise ValueError(
                f"Missing country column in {source_name}; pass country=... or include a country column."
            )
        data["country"] = country

    required = {"country", period_column}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing required columns in {source_name}: {sorted(missing)}")

    data = data.rename(columns={count_column: "total_news_articles"})
    data[period_column] = pd.to_datetime(data[period_column])
    data["country"] = data["country"].astype(str).replace(FILENAME_COUNTRY_MAP)
    data = data[["country", period_column, "total_news_articles"]].copy()
    data["total_news_articles"] = pd.to_numeric(data["total_news_articles"], errors="coerce")
    data = data[data["total_news_articles"].notna()].copy()
    data["total_news_articles"] = data["total_news_articles"].astype(int)
    data["country"] = pd.Categorical(data["country"], COUNTRY_ORDER, ordered=True)
    return data


def load_total_coverage_local(path, period_column):
    data = pd.read_csv(path)
    return standardize_total_coverage(data, period_column, source_name=str(path))


def load_weekly_total_coverage_from_mount(folder):
    files = sorted(folder.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in mounted folder: {folder}")

    frames = []
    for path in files:
        country = country_from_filename(path.name)
        print(f"Reading: {path}")
        data = pd.read_csv(path)
        data = standardize_total_coverage(data, "week", source_name=str(path), country=country)
        data["source_file"] = path.name
        frames.append(data)
        print(f"Loaded {country}: {len(data):,} rows")

    combined = pd.concat(frames, ignore_index=True)
    combined = (
        combined.groupby(["country", "week"], observed=True)["total_news_articles"]
        .sum()
        .reset_index()
    )
    return combined


def list_rd_csvs(rd_dir):
    from rd_io import rd_list_dir

    files = rd_list_dir(rd_dir)
    return sorted(file for file in files if file.lower().endswith(".csv"))


def read_rd_csv(rd_dir, filename):
    from rd_io import rd_join, rd_read_csv_df

    return rd_read_csv_df(rd_join(rd_dir, filename))


def load_weekly_total_coverage_from_webdav(rd_dir):
    files = list_rd_csvs(rd_dir)
    if not files:
        raise FileNotFoundError(f"No CSV files found in WebDAV folder: {rd_dir}")

    print("WebDAV total-coverage CSV files found:")
    for filename in files:
        print(f"- {filename}")

    weekly_files = [filename for filename in files if "week" in filename.lower()]
    selected_files = weekly_files or files

    frames = []
    for filename in selected_files:
        country = country_from_filename(filename)
        data = read_rd_csv(rd_dir, filename)
        data = standardize_total_coverage(data, "week", source_name=filename, country=country)
        data["source_file"] = filename
        frames.append(data)

    combined = pd.concat(frames, ignore_index=True)
    combined = (
        combined.groupby(["country", "week"], observed=True)["total_news_articles"]
        .sum()
        .reset_index()
    )
    return combined


# Load weekly total-news denominator.
if TOTAL_WEEK_PATH.exists():
    total_week = load_total_coverage_local(TOTAL_WEEK_PATH, "week")
    print(f"Loaded local cached weekly total-news denominator: {TOTAL_WEEK_PATH}")
elif TOTAL_COVERAGE_MOUNT_DIR.exists():
    print(f"Loading weekly total-news denominator from mounted WebDAV: {TOTAL_COVERAGE_MOUNT_DIR}")
    total_week = load_weekly_total_coverage_from_mount(TOTAL_COVERAGE_MOUNT_DIR)
    total_week.to_csv(TOTAL_WEEK_PATH, index=False)
    print(f"Cached mounted weekly denominator locally: {TOTAL_WEEK_PATH}")
else:
    print(f"Local weekly total-news denominator missing: {TOTAL_WEEK_PATH}")
    print(f"Mounted WebDAV folder missing: {TOTAL_COVERAGE_MOUNT_DIR}")
    print("Trying WebDAV API weekly total-coverage folder...")
    total_week = load_weekly_total_coverage_from_webdav(TOTAL_COVERAGE_RD_DIR)
    total_week.to_csv(TOTAL_WEEK_PATH, index=False)
    print(f"Cached WebDAV weekly denominator locally: {TOTAL_WEEK_PATH}")

# Load or derive monthly total-news denominator.
if TOTAL_MONTH_PATH.exists():
    total_month = load_total_coverage_local(TOTAL_MONTH_PATH, "month")
    print(f"Loaded local cached monthly total-news denominator: {TOTAL_MONTH_PATH}")
else:
    total_month = total_week.copy()
    total_month["month"] = total_month["week"].dt.to_period("M").dt.to_timestamp()
    total_month = (
        total_month.groupby(["country", "month"], observed=True)["total_news_articles"]
        .sum()
        .reset_index()
    )
    total_month.to_csv(TOTAL_MONTH_PATH, index=False)
    print(f"Derived and cached monthly denominator locally: {TOTAL_MONTH_PATH}")

print("Monthly total-news denominator:", total_month.shape)
display(total_month.head())
print("Weekly total-news denominator:", total_week.shape)
display(total_week.head())

## 3. Load Classified Political-Corruption Articles

These classifier outputs are the numerator: articles predicted to be primarily about political corruption.

In [ ]:
use_columns = {"country", "date_parsed", "year", "month", "week", "pred_political_corruption"}
frames = []

for country in COUNTRY_ORDER:
    path = CLASSIFIED_DIR / f"{country}_classified.csv.gz"
    if not path.exists():
        print(f"Missing classified file: {path}")
        continue

    data = pd.read_csv(path, usecols=lambda column: column in use_columns)
    data["country"] = country
    data = data[data["pred_political_corruption"].eq(1)].copy()
    frames.append(data)
    print(f"Loaded {country}: {len(data):,} political-corruption articles")

if not frames:
    raise FileNotFoundError(f"No classified files found in {CLASSIFIED_DIR}")

pc_articles = pd.concat(frames, ignore_index=True)
pc_articles["date_parsed"] = pd.to_datetime(pc_articles["date_parsed"], errors="coerce", utc=True)
pc_articles["date_naive"] = pc_articles["date_parsed"].dt.tz_convert(None)

if "month" in pc_articles.columns:
    pc_articles["month"] = pd.to_datetime(pc_articles["month"], errors="coerce")
else:
    pc_articles["month"] = pc_articles["date_naive"].dt.to_period("M").dt.to_timestamp()

if "week" in pc_articles.columns:
    pc_articles["week"] = pd.to_datetime(pc_articles["week"], errors="coerce")
else:
    pc_articles["week"] = pc_articles["date_naive"].dt.to_period("W").dt.start_time

pc_articles["country"] = pd.Categorical(pc_articles["country"], COUNTRY_ORDER, ordered=True)

print(f"Total political-corruption articles loaded: {len(pc_articles):,}")
display(pc_articles.head())

## 4. Build Weekly And Monthly Attention Tables

`relative_attention` is the percentage of **all news articles** classified as political corruption in a country-period.

In [ ]:
pc_week = (
    pc_articles.groupby(["country", "week"], observed=True)
    .size()
    .reset_index(name="political_corruption_articles")
)

pc_month = (
    pc_articles.groupby(["country", "month"], observed=True)
    .size()
    .reset_index(name="political_corruption_articles")
)

attention_month = total_month.merge(pc_month, on=["country", "month"], how="left")
attention_month["political_corruption_articles"] = attention_month["political_corruption_articles"].fillna(0).astype(int)
attention_month["relative_attention"] = attention_month["political_corruption_articles"] / attention_month["total_news_articles"]
attention_month["relative_attention_pct"] = attention_month["relative_attention"] * 100
attention_month["country_label"] = attention_month["country"].astype(str).replace(COUNTRY_LABELS)

if total_week is not None:
    attention_week = total_week.merge(pc_week, on=["country", "week"], how="left")
    attention_week["political_corruption_articles"] = attention_week["political_corruption_articles"].fillna(0).astype(int)
    attention_week["relative_attention"] = attention_week["political_corruption_articles"] / attention_week["total_news_articles"]
    attention_week["relative_attention_pct"] = attention_week["relative_attention"] * 100
    attention_week["country_label"] = attention_week["country"].astype(str).replace(COUNTRY_LABELS)
    attention_week.to_csv(TABLE_DIR / "political_corruption_attention_total_news_week.csv", index=False)
else:
    attention_week = None

attention_month.to_csv(TABLE_DIR / "political_corruption_attention_total_news_month.csv", index=False)

print(f"Saved attention tables to: {TABLE_DIR}")
display(attention_month.head())

## 5. Overall Summary

In [ ]:
country_summary = (
    attention_month.groupby(["country", "country_label"], observed=True)
    .agg(
        total_news_articles=("total_news_articles", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
country_summary["relative_attention"] = (
    country_summary["political_corruption_articles"] / country_summary["total_news_articles"]
)
country_summary["relative_attention_pct"] = country_summary["relative_attention"] * 100
country_summary = country_summary.sort_values("relative_attention_pct", ascending=False)

overall_total = country_summary["total_news_articles"].sum()
overall_pc = country_summary["political_corruption_articles"].sum()
overall_rate = overall_pc / overall_total

print(f"Total news articles:                   {overall_total:,}")
print(f"Total political-corruption articles:   {overall_pc:,}")
print(f"Overall relative attention:            {overall_rate:.4%}")

country_summary.to_csv(TABLE_DIR / "political_corruption_attention_total_news_country_summary.csv", index=False)
display(country_summary)

## 6. LaTeX Summary Tables

These tables summarize political-corruption attention relative to total news coverage. They are written as standalone LaTeX table files for the manuscript/appendix.

In [ ]:
ATTENTION_LATEX_DIR = TABLE_DIR / "latex"
ATTENTION_LATEX_DIR.mkdir(parents=True, exist_ok=True)


def resize_latex_tabular(latex):
    begin = "\\begin{tabular}"
    end = "\\end{tabular}"
    if begin not in latex or end not in latex:
        return latex
    latex = latex.replace(begin, "\\resizebox{\\textwidth}{!}{%\n" + begin, 1)
    latex = latex.replace(end, end + "\n}", 1)
    return latex


def add_table_note(latex, note):
    return latex.replace("\\end{table}\n", f"\\par\\smallskip\\footnotesize{{{note}}}\n\\end{{table}}\n")


def save_attention_latex_table(dataframe, filename, caption, label, note, resize=True):
    path = ATTENTION_LATEX_DIR / filename
    latex = dataframe.to_latex(
        index=False,
        escape=True,
        caption=caption,
        label=label,
        float_format="%.3f",
        bold_rows=False,
    )
    if resize:
        latex = resize_latex_tabular(latex)
    latex = add_table_note(latex, note)
    path.write_text(latex)
    print(f"Saved {path}")
    return path


summary_table = country_summary.copy()
summary_table["share_of_pc_articles_pct"] = (
    summary_table["political_corruption_articles"]
    / summary_table["political_corruption_articles"].sum()
    * 100
)
summary_table = summary_table[
    [
        "country_label",
        "total_news_articles",
        "political_corruption_articles",
        "relative_attention_pct",
        "share_of_pc_articles_pct",
    ]
].rename(
    columns={
        "country_label": "Country",
        "total_news_articles": "Total news",
        "political_corruption_articles": "PC articles",
        "relative_attention_pct": "PC share of total news (%)",
        "share_of_pc_articles_pct": "Share of PC corpus (%)",
    }
)

save_attention_latex_table(
    summary_table,
    "table_attention_country_summary.tex",
    "Political-corruption attention by country relative to total news coverage.",
    "tab:pc-attention-country-summary",
    "Note. PC = political corruption. Relative attention is the percentage of total news coverage classified as political corruption.",
)
display(summary_table)


yearly_attention_for_tables = attention_month.copy()
yearly_attention_for_tables["year"] = yearly_attention_for_tables["month"].dt.year
yearly_attention_for_tables = (
    yearly_attention_for_tables.groupby(["country", "country_label", "year"], observed=True)
    .agg(
        total_news_articles=("total_news_articles", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
yearly_attention_for_tables["relative_attention_pct"] = (
    yearly_attention_for_tables["political_corruption_articles"]
    / yearly_attention_for_tables["total_news_articles"]
    * 100
)

overall_year_table = (
    yearly_attention_for_tables.groupby("year", observed=True)
    .agg(
        total_news_articles=("total_news_articles", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
overall_year_table["relative_attention_pct"] = (
    overall_year_table["political_corruption_articles"]
    / overall_year_table["total_news_articles"]
    * 100
)
overall_year_latex = overall_year_table.rename(
    columns={
        "year": "Year",
        "total_news_articles": "Total news",
        "political_corruption_articles": "PC articles",
        "relative_attention_pct": "PC share of total news (%)",
    }
)

save_attention_latex_table(
    overall_year_latex,
    "table_attention_year_summary.tex",
    "Political-corruption attention by year across all countries.",
    "tab:pc-attention-year-summary",
    "Note. PC = political corruption. Counts are summed across countries.",
)
display(overall_year_latex)


country_year_matrix = yearly_attention_for_tables.pivot(
    index="country_label",
    columns="year",
    values="relative_attention_pct",
).reindex([COUNTRY_LABELS.get(country, country) for country in COUNTRY_ORDER])
country_year_matrix = country_year_matrix.reset_index().rename(columns={"country_label": "Country"})

save_attention_latex_table(
    country_year_matrix,
    "table_attention_country_year_matrix.tex",
    "Political-corruption attention by country-year as a percentage of total news coverage.",
    "tab:pc-attention-country-year-matrix",
    "Note. Entries are percentages of total news coverage classified as political corruption.",
)
display(country_year_matrix)


peak_months = attention_month.copy()
peak_months = peak_months[peak_months["total_news_articles"].gt(0)].copy()
peak_months["rank"] = peak_months.groupby("country", observed=True)["relative_attention_pct"].rank(
    method="first",
    ascending=False,
)
peak_months = peak_months[peak_months["rank"].le(3)].copy()
peak_months["month_label"] = peak_months["month"].dt.strftime("%Y-%m")
peak_months = peak_months.sort_values(["country", "rank"])
peak_months_table = peak_months[
    [
        "country_label",
        "rank",
        "month_label",
        "total_news_articles",
        "political_corruption_articles",
        "relative_attention_pct",
    ]
].rename(
    columns={
        "country_label": "Country",
        "rank": "Rank",
        "month_label": "Month",
        "total_news_articles": "Total news",
        "political_corruption_articles": "PC articles",
        "relative_attention_pct": "PC share of total news (%)",
    }
)

save_attention_latex_table(
    peak_months_table,
    "table_attention_peak_months.tex",
    "Peak months of political-corruption attention by country.",
    "tab:pc-attention-peak-months",
    "Note. PC = political corruption. Peak months are ranked within country by relative attention.",
)
display(peak_months_table)

print(f"LaTeX attention tables written to: {ATTENTION_LATEX_DIR}")


## 7. Optional Diagnostic: Share Within Corruption-Query Corpus

This is **not** the main attention denominator. It is only a diagnostic showing how restrictive the political-corruption classifier is within the keyword-collected corruption-query corpus.

In [ ]:
query_month_path = PIPELINE_DIR / "denominator_country_month.csv"
if query_month_path.exists():
    query_month = pd.read_csv(query_month_path)
    query_month["month"] = pd.to_datetime(query_month["month"])
    query_month = query_month.rename(columns={"total_articles": "corruption_query_articles"})
    query_month["country"] = pd.Categorical(query_month["country"].astype(str), COUNTRY_ORDER, ordered=True)
    query_attention_month = query_month.merge(pc_month, on=["country", "month"], how="left")
    query_attention_month["political_corruption_articles"] = query_attention_month["political_corruption_articles"].fillna(0).astype(int)
    query_attention_month["share_within_corruption_query"] = (
        query_attention_month["political_corruption_articles"] / query_attention_month["corruption_query_articles"]
    )
    query_attention_month.to_csv(TABLE_DIR / "political_corruption_share_within_corruption_query_month.csv", index=False)
    display(query_attention_month.head())
else:
    print(f"No corruption-query diagnostic denominator found at {query_month_path}")

## 8. Plot Helpers

In [ ]:
COUNTRY_COLORS = {
    "Bulgaria": "#3B6FB6",
    "France": "#E6862E",
    "Hungary": "#C83E4D",
    "Italy": "#2A9D8F",
    "Netherlands": "#5C9E3F",
    "Serbia": "#C9A227",
    "Sweden": "#8E6BBE",
    "Ukraine": "#D95F8D",
    "United_Kingdom": "#8A6A55",
}


def period_data(level=PLOT_LEVEL):
    if level == "week":
        if attention_week is None:
            raise ValueError("Weekly plot requested, but no weekly total-news denominator was loaded.")
        data = attention_week.copy()
        period = "week"
        label = "Weekly"
    elif level == "month":
        data = attention_month.copy()
        period = "month"
        label = "Monthly"
    else:
        raise ValueError("level must be 'week' or 'month'")
    return data, period, label


def add_rolling_average(data, period, window=3):
    data = data.sort_values(["country", period]).copy()
    data[f"relative_attention_pct_roll{window}"] = (
        data.groupby("country", observed=True)["relative_attention_pct"]
        .transform(lambda series: series.rolling(window=window, min_periods=1).mean())
    )
    return data


def save_figure(fig, name):
    png = FIGURE_DIR / f"{name}.png"
    pdf = FIGURE_DIR / f"{name}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved {png}")
    print(f"Saved {pdf}")


def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=0)
    return ax


def percent_axis(ax):
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
    return ax


def add_source_note(fig, text="Relative attention = predicted political-corruption articles / total news articles."):
    fig.text(0.01, 0.01, text, ha="left", va="bottom", fontsize=8, color="#555555")


## 9. Relative Attention Over Time By Country

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

fig, ax = plt.subplots(figsize=(12, 6))

for country in COUNTRY_ORDER:
    country_data = data[data["country"].astype(str).eq(country)].sort_values(period)
    if country_data.empty:
        continue
    ax.plot(
        country_data[period],
        country_data["relative_attention_pct"],
        label=COUNTRY_LABELS.get(country, country),
        color=COUNTRY_COLORS[country],
        linewidth=1.8,
        alpha=0.9,
    )

ax.set_title(f"{level_label} Relative Attention To Political Corruption By Country")
ax.set_ylabel("Political-corruption articles (% of total news coverage)")
ax.set_xlabel("")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_ylim(bottom=0)
fig.tight_layout()
save_figure(fig, f"political_corruption_relative_attention_total_news_{period}_country_lines")

## 10. Small Multiples

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, country in zip(axes, COUNTRY_ORDER):
    country_data = data[data["country"].astype(str).eq(country)].sort_values(period)
    ax.plot(
        country_data[period],
        country_data["relative_attention_pct"],
        color=COUNTRY_COLORS[country],
        linewidth=1.8,
    )
    ax.fill_between(
        country_data[period],
        country_data["relative_attention_pct"],
        color=COUNTRY_COLORS[country],
        alpha=0.16,
    )
    ax.set_title(COUNTRY_LABELS.get(country, country), loc="left", fontweight="bold")
    format_time_axis(ax)
    ax.set_ylim(bottom=0)

fig.suptitle(f"{level_label} Relative Attention To Political Corruption", y=1.02, fontsize=14)
fig.text(0.5, -0.01, "Year", ha="center")
fig.text(0.0, 0.5, "% of total news coverage", va="center", rotation="vertical")
fig.tight_layout()
save_figure(fig, f"political_corruption_relative_attention_total_news_{period}_small_multiples")

## 11. Absolute Political-Corruption Volume

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

wide_counts = (
    data.pivot_table(
        index=period,
        columns="country",
        values="political_corruption_articles",
        aggfunc="sum",
        fill_value=0,
        observed=True,
    )
    .reindex(columns=COUNTRY_ORDER)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(
    wide_counts.index,
    [wide_counts[country].to_numpy() for country in COUNTRY_ORDER],
    labels=[COUNTRY_LABELS.get(country, country) for country in COUNTRY_ORDER],
    colors=[COUNTRY_COLORS[country] for country in COUNTRY_ORDER],
    alpha=0.88,
)
ax.set_title(f"{level_label} Volume Of Political-Corruption Coverage")
ax.set_ylabel("Predicted political-corruption articles")
ax.set_xlabel("")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
save_figure(fig, f"political_corruption_absolute_volume_{period}_stacked")

## 12. Country-Year Heatmap

In [ ]:
yearly_attention = attention_month.copy()
yearly_attention["year"] = yearly_attention["month"].dt.year
yearly_attention = (
    yearly_attention.groupby(["country", "country_label", "year"], observed=True)
    .agg(
        total_news_articles=("total_news_articles", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
yearly_attention["relative_attention_pct"] = (
    yearly_attention["political_corruption_articles"] / yearly_attention["total_news_articles"] * 100
)

heatmap_data = yearly_attention.pivot(index="country_label", columns="year", values="relative_attention_pct")
heatmap_data = heatmap_data.reindex([COUNTRY_LABELS.get(country, country) for country in COUNTRY_ORDER])

fig, ax = plt.subplots(figsize=(11, 5.5))
image = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd")
ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns.astype(int))
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Relative Attention To Political Corruption By Country-Year")

for y in range(heatmap_data.shape[0]):
    for x in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[y, x]
        if pd.notna(value):
            ax.text(x, y, f"{value:.2f}", ha="center", va="center", fontsize=7, color="black")

cbar = fig.colorbar(image, ax=ax)
cbar.set_label("Political-corruption articles (% of total news coverage)")
fig.tight_layout()
save_figure(fig, "political_corruption_relative_attention_total_news_country_year_heatmap")

yearly_attention.to_csv(TABLE_DIR / "political_corruption_attention_total_news_country_year.csv", index=False)
display(yearly_attention.head())

## 13. Country Ranking And Indexed Attention

These figures give a cleaner cross-country summary for talks or manuscript drafts.

In [ ]:
ranking = country_summary.sort_values("relative_attention_pct", ascending=True).copy()

fig, ax = plt.subplots(figsize=(9, 5.8))
colors = [COUNTRY_COLORS.get(country, "#777777") for country in ranking["country"].astype(str)]
ax.barh(ranking["country_label"], ranking["relative_attention_pct"], color=colors, alpha=0.92)
ax.set_title("Average Attention To Political Corruption By Country")
ax.set_xlabel("Political-corruption articles (% of total news coverage)")
ax.set_ylabel("")
percent_axis(ax)
for y, value in enumerate(ranking["relative_attention_pct"]):
    ax.text(value + ranking["relative_attention_pct"].max() * 0.01, y, f"{value:.2f}%", va="center", fontsize=8)
ax.set_xlim(0, ranking["relative_attention_pct"].max() * 1.18)
add_source_note(fig)
fig.tight_layout(rect=(0, 0.04, 1, 1))
save_figure(fig, "political_corruption_attention_country_ranking_total_news")


In [ ]:
data, period, level_label = period_data("month")
indexed = add_rolling_average(data, "month", window=3)
indexed["country_mean_attention"] = indexed.groupby("country", observed=True)["relative_attention_pct"].transform("mean")
indexed["attention_index"] = indexed["relative_attention_pct_roll3"] / indexed["country_mean_attention"] * 100

fig, ax = plt.subplots(figsize=(12, 6))
for country in COUNTRY_ORDER:
    country_data = indexed[indexed["country"].astype(str).eq(country)].sort_values("month")
    ax.plot(
        country_data["month"],
        country_data["attention_index"],
        label=COUNTRY_LABELS.get(country, country),
        color=COUNTRY_COLORS[country],
        linewidth=1.7,
        alpha=0.88,
    )
ax.axhline(100, color="#222222", linewidth=1.0, linestyle="--", alpha=0.7)
ax.set_title("Indexed Political-Corruption Attention Over Time")
ax.set_ylabel("Index, country average = 100")
ax.set_xlabel("")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
add_source_note(fig, "Three-month rolling average; each country indexed to its own mean attention level.")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save_figure(fig, "political_corruption_attention_monthly_indexed_country_lines")


## 14. What To Report

For a manuscript figure, start with the monthly small-multiple relative-attention plot. Use the stacked absolute-volume plot as a secondary/descriptive figure, and the country-year heatmap as an appendix-friendly overview.

Remember to describe the denominator precisely: the relative-attention figures divide predicted political-corruption articles by **total news coverage** in each country-period.